# Progetto 2 – Parte 2: LLM via Ollama e confronto con la GNN

Domanda di ricerca (1): **per quali valori di n la timed GNN guida meglio di un LLM?**

L'ipotesi della traccia è che la GNN sia migliore per n piccoli (1–3) e l'LLM per n grandi.

Per avere un confronto equo l'LLM viene valutato **esattamente come la GNN**:
- stessi puzzle di test (`test_puzzles.parquet`);
- stessa funzione `solve()`: l'LLM propone la mossa, la difesa risponde con la soluzione oppure con Stockfish se l'LLM ha deviato, e il puzzle è risolto se arriva il matto entro n mosse.

Scelte fatte:
- **zero-shot**, come da traccia: niente esempi e niente fine-tuning;
- l'LLM vede la posizione **nell'orientamento originale** (non quella specchiata usata per la GNN);
- gli diamo la **lista delle mosse legali**: la GNN sceglie solo tra mosse legali, quindi così il confronto è alla pari (si può disattivare con `SHOW_LEGAL`);
- `temperature = 0` e seed fisso: le risposte sono ripetibili, quindi basta una sola run per modello.

## Setup di Ollama

Da terminale:
```bash
brew install ollama
ollama serve                 # lasciarlo aperto (oppure avviare l'app Ollama)
ollama pull llama3.1:8b      # in un altro terminale
ollama pull qwen3:8b
```

Con 24 GB di RAM vanno bene modelli fino a circa 14B parametri. `llama3.1:8b` rappresenta la famiglia Llama citata nella traccia; `qwen3:8b` è un modello più recente. Opzionale: `gpt-oss:20b`, ma ragiona sempre (non si può spegnere) ed è molto più lento.

In [ ]:
import os, re, json, time, glob, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import chess, chess.engine
from utils import solve, mirror

cfg = json.load(open("config.json"))          # stesso file della parte 1
DATA, RUNS = cfg["data_dir"], cfg["runs_dir"]

OLLAMA = "http://localhost:11434/api/chat"
# modello -> parametro "think" (None = non lo mandiamo, False = ragionamento interno spento)
MODELS = {
    "llama3.1:8b": None,
    "qwen3:8b": False,
}
LLM_PER_N = 100        # puzzle per n (l'LLM è lento: 100 x 5 = 500 puzzle per modello)
SHOW_LEGAL = True      # dare all'LLM la lista delle mosse legali
MAX_TOKENS = 400       # tetto alla risposta: senza, i modelli piccoli possono andare in loop
GNN = "M4"             # la timed GNN da confrontare (M4 = time-decay con tempo)

print("modelli installati:", [m["name"] for m in requests.get("http://localhost:11434/api/tags").json()["models"]])

## 1. Puzzle di test

Prendiamo i primi `LLM_PER_N` puzzle per ogni n da `test_puzzles.parquet`: sono un sottoinsieme degli stessi 1.500 puzzle usati per la GNN, quindi il confronto è appaiato puzzle per puzzle.

Con 100 puzzle per n e circa 5–10 secondi per risposta, ogni modello impiega circa 1–2 ore. Il ciclo salva dopo ogni puzzle, quindi si può interrompere e riprendere.

In [ ]:
test_p = pd.read_parquet(f"{DATA}/test_puzzles.parquet")
llm_test = test_p.groupby("n").head(LLM_PER_N)
print(len(llm_test), "puzzle:", llm_test["n"].value_counts().sort_index().to_dict())

## 2. Il prompt

È quello della traccia ("Given this FEN, find mate in n moves. Explain step-by-step"), con tre aggiunte:
- la scacchiera disegnata, perché gli LLM leggono male il FEN da solo;
- la lista delle mosse legali;
- la richiesta di rispondere in JSON (`reasoning` + `move`).

Il prompt è in inglese perché gli LLM rendono meglio in inglese sugli scacchi.

Nel JSON chiediamo prima il ragionamento e poi la mossa: così il modello "pensa" prima di decidere. Il formato è imposto da Ollama con uno schema (`format`), quindi la risposta è sempre un JSON valido.

In [ ]:
def board_text(board):
    rows = str(board).split("\n")
    return "\n".join(f"{8 - i} {r}" for i, r in enumerate(rows)) + "\n  a b c d e f g h"


def make_prompt(board, moves_left):
    side = "White" if board.turn == chess.WHITE else "Black"
    text = (f"You are a chess expert. Solve this chess puzzle.\n\n"
            f"FEN: {board.fen()}\n\n"
            f"Board (uppercase = White, lowercase = Black):\n{board_text(board)}\n\n"
            f"{side} to move. {side} can deliver checkmate in {moves_left} move(s) or fewer.\n")
    if SHOW_LEGAL:
        text += "Legal moves (UCI): " + " ".join(m.uci() for m in board.legal_moves) + "\n"
    text += ("\nThink step by step, then choose the best move. Answer in JSON with two fields: "
             "\"reasoning\" (a short step-by-step explanation, at most 5 sentences) and "
             "\"move\" (the move in UCI format, e.g. e2e4).")
    return text


p = llm_test.iloc[0]
b = chess.Board(p["fen_orig"]); b.push_uci(p["moves_orig"].split()[0])
print(make_prompt(b, p["n"]))

## 3. Chiamata a Ollama e lettura della mossa

`ask_llm` manda il prompt e restituisce mossa, ragionamento e token generati.

`parse_move` accetta sia il formato UCI (`e2e4`) sia quello algebrico (`Qh7#`), perché i modelli a volte ignorano l'istruzione, e toglie eventuali numeri di mossa (`1. e4`). Se la mossa non è legale restituisce `None`.

In [ ]:
SCHEMA = {"type": "object",
          "properties": {"reasoning": {"type": "string"}, "move": {"type": "string"}},
          "required": ["reasoning", "move"]}


def ask_llm(model, prompt):
    think = MODELS[model]
    body = {"model": model, "stream": False, "format": SCHEMA,
            "messages": [{"role": "user", "content": prompt}],
            "options": {"temperature": 0, "seed": 0, "num_ctx": 8192,
                        "num_predict": MAX_TOKENS if think in (None, False) else 4000}}
    if think is not None:
        body["think"] = think
    r = requests.post(OLLAMA, json=body, timeout=900).json()
    if "error" in r:
        raise RuntimeError(r["error"])
    try:
        ans = json.loads(r["message"]["content"])
    except json.JSONDecodeError:            # risposta troncata da MAX_TOKENS
        ans = {"reasoning": r["message"]["content"], "move": ""}
    return str(ans.get("move", "")), str(ans.get("reasoning", "")), r.get("eval_count", 0)


def parse_move(board, text):
    text = re.sub(r"^\d+\.+\s*", "", text.strip())        # toglie "1." o "1..."
    try:
        m = chess.Move.from_uci(text.lower())
        if m in board.legal_moves:
            return m.uci()
    except ValueError:
        pass
    try:
        return board.parse_san(text).uci()                   # es. "Qh7#"
    except ValueError:
        return None

## 4. L'LLM come giocatore per `solve()`

`solve()` lavora in forma canonica: il risolutore gioca col Bianco. `choose` quindi:
1. specchia la scacchiera se il puzzle era specchiato, così l'LLM vede la posizione reale;
2. chiede la mossa, aggiornando il numero di mosse rimaste a ogni turno;
3. riporta la mossa in forma canonica.

Se l'LLM dà una mossa illegale o illeggibile il puzzle finisce con `status = "illegal"` (contato come non risolto). Ogni risposta viene salvata in `log` per l'analisi.

In [ ]:
class Illegal(Exception):
    pass


def make_choose(model, log):
    def choose(p, board, played):
        real = board.mirror() if p["flip"] else board
        moves_left = p["n"] - (len(played) - 1) // 2
        answer, reasoning, tokens = ask_llm(model, make_prompt(real, moves_left))
        m = parse_move(real, answer)
        log.append({"answer": answer, "move": m, "reasoning": reasoning, "tokens": tokens})
        if m is None:
            raise Illegal()
        return mirror(m) if p["flip"] else m
    return choose


# prova su un puzzle
model = list(MODELS)[0]
log = []
p = llm_test.iloc[0].to_dict()
try:
    print(solve(p, make_choose(model, log), None)[0])
except Illegal:
    print("illegal")
for l in log:
    print(f"- mossa: {l['answer']!r} ({l['tokens']} token)\n  {l['reasoning']}")
print("soluzione (originale):", p["moves_orig"])

## 5. Valutazione

Per ogni puzzle salviamo, nello stesso formato della GNN:
- l'esito e se la prima mossa era giusta (`first_ok`);
- il tempo, il numero di chiamate e i token;
- le risposte e il ragionamento della prima mossa, che serve per la valutazione umana.

I risultati vanno in `runs/LLM_<modello>/results.csv`, una riga alla volta: se si interrompe, rilanciando la cella si riparte da dove si era arrivati.

In [ ]:
engine = chess.engine.SimpleEngine.popen_uci(cfg["stockfish"]) if cfg["stockfish"] else None

for model in MODELS:
    name = "LLM_" + model.replace(":", "_").replace("/", "_")
    out = f"{RUNS}/{name}/results.csv"
    os.makedirs(os.path.dirname(out), exist_ok=True)
    done = set(pd.read_csv(out)["case"]) if os.path.exists(out) else set()
    todo = llm_test[~llm_test["case"].isin(done)]
    print(f"{model}: {len(done)} già fatti, {len(todo)} da fare")
    t_start = time.time()
    for i, p in enumerate(todo.to_dict("records")):
        log, t0 = [], time.time()
        try:
            status, _, played = solve(p, make_choose(model, log), engine)
        except Illegal:
            status, played = "illegal", []
        first = log[0]["move"] if log else None
        if first and p["flip"]:
            first = mirror(first)
        row = {"system": name, "seed": 0, "case": p["case"], "n": p["n"], "status": status,
               "solved": status == "solved", "first_ok": first == p["moves"].split()[1],
               "moves": " ".join(played), "sec": time.time() - t0, "calls": len(log),
               "tokens": sum(l["tokens"] for l in log),
               "answers": json.dumps([l["answer"] for l in log]),
               "reasoning": log[0]["reasoning"] if log else ""}
        pd.DataFrame([row]).to_csv(out, mode="a", header=not os.path.exists(out), index=False)
        if (i + 1) % 10 == 0:
            el = time.time() - t_start
            print(f"  {i + 1}/{len(todo)}  ({el / (i + 1):.1f} s/puzzle, mancano ~{el / (i + 1) * (len(todo) - i - 1) / 60:.0f} min)")
if engine:
    engine.quit()

## 6. Risultati: chi risolve di più, per n

Tutti i sistemi vengono valutati sugli stessi puzzle del sottoinsieme LLM. Per la GNN si riporta la media dei 3 seed.

In [ ]:
res = pd.concat([pd.read_csv(f) for f in glob.glob(f"{RUNS}/*/results.csv")], ignore_index=True)
res = res[res["case"].isin(llm_test["case"])]
llm_names = sorted(s for s in res["system"].unique() if s.startswith("LLM_"))
show = ["M1", GNN] + llm_names + ["greedy", "random"]
show = list(dict.fromkeys(show))      # toglie doppioni se GNN = M1

table = (res[res.system.isin(show)].groupby(["system", "seed", "n"])["solved"].mean()
         .groupby(["system", "n"]).mean().unstack().loc[show].round(3))
table["tot"] = res[res.system.isin(show)].groupby("system")["solved"].mean().loc[show].round(3)
table

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for s in show:
    ax.plot(table.columns[:-1], table.loc[s].iloc[:-1], marker="o", label=s)
ax.set_xlabel("n (matto in n)"); ax.set_ylabel("puzzle risolti"); ax.set_xticks(range(1, 6))
ax.set_title("GNN vs LLM: tasso di soluzione per n"); ax.legend(fontsize=8)
plt.show()

In [ ]:
# prima mossa giusta, mosse illegali, tempo per puzzle
first = res[res.system.isin(show)].groupby(["system", "n"])["first_ok"].mean().unstack().loc[show].round(3)
print("Prima mossa corretta:"); display(first)

llm = res[res.system.isin(llm_names)]
print("LLM: quota di puzzle finiti con mossa illegale/illeggibile")
display(llm.groupby(["system", "n"])["status"].apply(lambda s: (s == "illegal").mean()).unstack().round(3))
print("Secondi per puzzle (efficienza):")
display(res[res.system.isin(show) & res["sec"].notna()].groupby("system")["sec"].agg(["mean", "median"]).round(2))

## 7. Test statistico: GNN vs LLM per n

Per ogni n confrontiamo la GNN con ciascun LLM sugli stessi puzzle (test di McNemar).

Per la GNN usiamo il **voto di maggioranza dei 3 seed**: un puzzle conta come risolto se lo risolvono almeno 2 seed su 3. Così otteniamo un esito sì/no per puzzle, come per l'LLM.

Il vincitore viene indicato solo se p < 0,05.

In [ ]:
from scipy.stats import binomtest

gnn = res[res.system == GNN].groupby("case")["solved"].mean() >= 2 / 3
rows = []
for name in llm_names:
    L = res[res.system == name].set_index("case")["solved"]
    for n in sorted(llm_test["n"].unique()):
        cases = llm_test.loc[llm_test["n"] == n, "case"]
        cases = cases[cases.isin(L.index)]
        g, l = gnn.loc[cases], L.loc[cases]
        only_g, only_l = int((g & ~l).sum()), int((~g & l).sum())
        p = binomtest(only_g, only_g + only_l).pvalue if only_g + only_l > 0 else 1.0
        win = "pari" if p >= 0.05 else ("GNN" if only_g > only_l else "LLM")
        rows.append({"LLM": name, "n": n, "puzzle": len(cases), "GNN risolti": round(g.mean(), 3),
                     "LLM risolti": round(l.mean(), 3), "solo GNN": only_g, "solo LLM": only_l,
                     "p": round(p, 4), "vince": win})
pd.DataFrame(rows)

## 8. Valutazione umana della guida (blind)

La traccia chiede anche di valutare la **qualità della guida** con un giudizio umano alla cieca. Questa cella prepara un foglio con 20 puzzle: per ognuno due guide anonime, A e B, in ordine casuale. Una viene dalla GNN (mossa consigliata, prima mossa del seed 0), l'altra dall'LLM (mossa più spiegazione).

I valutatori danno un voto da 1 a 5 per **correttezza** e **chiarezza**. La chiave per sapere chi è A e chi è B è in un file separato.

Limite da citare nel report: la GNN non produce spiegazioni, quindi il blind è solo parziale (la guida senza testo si riconosce). Il voto di correttezza resta comunque confrontabile.

In [ ]:
LLM_MAIN = llm_names[0]
random.seed(0)
sample = llm_test.sample(20, random_state=0)
g_res = res[(res.system == GNN) & (res.seed == 0)].set_index("case")
l_res = res[res.system == LLM_MAIN].set_index("case")


def san_of(p, canon_move):
    # mossa canonica -> notazione algebrica nella posizione originale
    if not isinstance(canon_move, str) or not canon_move:
        return "(nessuna mossa valida)"
    b = chess.Board(p["fen_orig"]); b.push_uci(p["moves_orig"].split()[0])
    m = chess.Move.from_uci(mirror(canon_move) if p["flip"] else canon_move)
    return b.san(m) if m in b.legal_moves else canon_move


sheet, key = [], []
for p in sample.to_dict("records"):
    g_moves = str(g_res.loc[p["case"], "moves"]).split()
    l_row = l_res.loc[p["case"]]
    l_moves = str(l_row["moves"]).split() if isinstance(l_row["moves"], str) else []
    guide_g = f"Mossa consigliata: {san_of(p, g_moves[1] if len(g_moves) > 1 else '')}"
    guide_l = f"Mossa consigliata: {san_of(p, l_moves[1] if len(l_moves) > 1 else '')}. {l_row['reasoning']}"
    b = chess.Board(p["fen_orig"]); b.push_uci(p["moves_orig"].split()[0])
    swap = random.random() < 0.5
    sheet.append({"puzzle": p["case"], "FEN": b.fen(), "n": p["n"],
                  "guida_A": guide_l if swap else guide_g, "guida_B": guide_g if swap else guide_l,
                  "A_correttezza": "", "A_chiarezza": "", "B_correttezza": "", "B_chiarezza": ""})
    key.append({"puzzle": p["case"], "A": LLM_MAIN if swap else GNN, "B": GNN if swap else LLM_MAIN})

pd.DataFrame(sheet).to_csv("valutazione_umana.csv", index=False)
pd.DataFrame(key).to_csv("valutazione_umana_CHIAVE.csv", index=False)
pd.DataFrame(sheet).head(3)

## Note per il report

- **Equità del confronto.** Stessi puzzle, stessa procedura di valutazione, mosse legali disponibili a entrambi i sistemi. L'LLM è zero-shot, la GNN è addestrata su circa 45.000 puzzle: è il confronto richiesto dalla traccia, ma va detto.
- **n da 6 a 10.** La traccia chiede n fino a 10 con problemi classici. La GNN è allenata su n = 1–5, quindi su n > 5 sarebbe fuori distribuzione; lo stesso codice funziona su qualsiasi file con le stesse colonne di `test_puzzles.parquet`.
- **Efficienza.** Confrontare i secondi per puzzle: la GNN impiega frazioni di secondo, un LLM locale molti secondi.
- **Varianti possibili.** `SHOW_LEGAL = False` (l'LLM senza aiuto, più vicino alla traccia) oppure un modello con ragionamento attivo (`"qwen3:8b": True`, molto più lento).